In [1]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import  OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.linear_model import Ridge
from sklearn.model_selection import cross_val_score, GridSearchCV

import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
# setting Jedha color palette as default
pio.templates["jedha"] = go.layout.Template(
    layout_colorway=["#4B9AC7", "#4BE8E0", "#9DD4F3", "#97FBF6", "#2A7FAF", "#23B1AB", "#0E3449", "#015955"]
)
pio.templates.default = "jedha"
pio.renderers.default = "svg" # to be replaced by "iframe" if working on JULIE

In [2]:
# Import dataset
print("Loading dataset...")
dataset = pd.read_csv("src/Walmart_Store_sales.csv")
print("...Done.")
print()

Loading dataset...
...Done.



In [3]:
# Basic stats
print("Number of rows : {}".format(dataset.shape[0]))
print()

print("Display of dataset: ")
display(dataset.head())
print()

print("Basics statistics: ")
data_desc = dataset.describe(include='all')
display(data_desc)
print()

print("Percentage of missing values: ")
display(100*dataset.isnull().sum()/dataset.shape[0])



Number of rows : 150

Display of dataset: 


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
0,6.0,18-02-2011,1572117.54,NaN,59.61,3.045,214.777523,6.858
1,13.0,25-03-2011,1807545.43,0.0,42.38,3.435,128.616064,7.470
2,17.0,27-07-2012,NaN,0.0,NaN,NaN,130.719581,5.936
3,11.0,NaN,1244390.03,0.0,84.57,NaN,214.556497,7.346
4,6.0,28-05-2010,1644470.66,0.0,78.89,2.759,212.412888,7.092



Basics statistics: 


,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,150.000000,132,1.360000e+02,138.000000,132.000000,136.000000,138.000000,135.000000
unique,NaN,85,NaN,NaN,NaN,NaN,NaN,NaN
top,NaN,07-01-2011,NaN,NaN,NaN,NaN,NaN,NaN
freq,NaN,4,NaN,NaN,NaN,NaN,NaN,NaN
mean,9.866667,NaN,1.249536e+06,0.079710,61.398106,3.320853,179.898509,7.598430
std,6.231191,NaN,6.474630e+05,0.271831,18.378901,0.478149,40.274956,1.577173
min,1.000000,NaN,2.689290e+05,0.000000,18.790000,2.514000,126.111903,5.143000
25%,4.000000,NaN,6.050757e+05,0.000000,45.587500,2.852250,131.970831,6.597500
50%,9.000000,NaN,1.261424e+06,0.000000,62.985000,3.451000,197.908893,7.470000
75%,15.750000,NaN,1.806386e+06,0.000000,76.345000,3.706250,214.934616,8.150000



Percentage of missing values: 


Store            0.000000
Date            12.000000
Weekly_Sales     9.333333
Holiday_Flag     8.000000
Temperature     12.000000
Fuel_Price       9.333333
CPI              8.000000
Unemployment    10.000000
dtype: float64

In [4]:
print("The following will be 'False' if there's no missing values in the dataset: ", dataset.isnull().any().any())


The following will be 'False' if there's no missing values in the dataset:  True


In [5]:
#gestion des valeurs manquantes#
print("Suppression des NaNs sur Weekly_Sales...")
dataset = dataset.dropna(subset=['Weekly_Sales'])

Suppression des NaNs sur Weekly_Sales...


In [6]:
#feature engineering temporel #
print("Extraction des features de la Date...")

dataset['Date'] = pd.to_datetime(dataset['Date'], format='%d-%m-%Y')
dataset['Year'] = dataset['Date'].dt.year
dataset['Month'] = dataset['Date'].dt.month
dataset['Day'] = dataset['Date'].dt.day
dataset['day_of_week'] = dataset['Date'].dt.dayofweek# 0=Lundi, 6=Dimanche
dataset['hour'] = dataset['Date'].dt.hour

Extraction des features de la Date...


In [7]:
# Suppression de la colonne d'origine#
dataset = dataset.drop(columns=['Date'])

In [8]:
print("Filtrage des Outliers (3 Ecarts-Types)...")
numeric_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

#filtrages des outliers(Règle des 3 écarts-types)
numeric_cols = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment']

for col in numeric_cols:
    mean_val = dataset[col].mean()
    std_val = dataset[col].std()
    
   # Garde les valeurs normales OU les valeurs manquantes
    mask = (dataset[col].isna()) | ((dataset[col] >= mean_val - 3 * std_val) & (dataset[col] <= mean_val + 3 * std_val))
    dataset = dataset[mask]     

print(f"Dimensions après nettoyage Pandas : {dataset.shape}")


Filtrage des Outliers (3 Ecarts-Types)...
Dimensions après nettoyage Pandas : (131, 12)


In [9]:
# Séparer la variable cible Y des caractéristiques X

print("Séparation des étiquettes et des caractéristiques...")
target_variable = 'Weekly_Sales'
X = dataset.drop(columns=[target_variable])
Y = dataset[target_variable]
print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Séparation des étiquettes et des caractéristiques...
...Done.

Y : 
0    1572117.54
1    1807545.43
3    1244390.03
4    1644470.66
5    1857533.70
Name: Weekly_Sales, dtype: float64

X :
   Store  Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment  \
0    6.0           NaN        59.61       3.045  214.777523         6.858   
1   13.0           0.0        42.38       3.435  128.616064         7.470   
3   11.0           0.0        84.57         NaN  214.556497         7.346   
4    6.0           0.0        78.89       2.759  212.412888         7.092   
5    4.0           0.0          NaN       2.756  126.160226         7.896   

     Year  Month   Day  day_of_week  hour  
0  2011.0    2.0  18.0          4.0   0.0  
1  2011.0    3.0  25.0          4.0   0.0  
3     NaN    NaN   NaN          NaN   NaN  
4  2010.0    5.0  28.0          4.0   0.0  
5  2010.0    5.0  28.0          4.0   0.0  


In [10]:
# Diviser le jeu de données en Train set & Test set
# Séparation de la variable cible Y des caractéristiques X
print("Séparation des étiquettes et des caractéristiques...")
target_variable = 'Weekly_Sales'

X = dataset.drop(columns=[target_variable])
Y = dataset[target_variable]

print("...Done.")
print()

print('Y : ')
print(Y.head())
print()
print('X :')
print(X.head())

Séparation des étiquettes et des caractéristiques...
...Done.

Y : 
0    1572117.54
1    1807545.43
3    1244390.03
4    1644470.66
5    1857533.70
Name: Weekly_Sales, dtype: float64

X :
   Store  Holiday_Flag  Temperature  Fuel_Price         CPI  Unemployment  \
0    6.0           NaN        59.61       3.045  214.777523         6.858   
1   13.0           0.0        42.38       3.435  128.616064         7.470   
3   11.0           0.0        84.57         NaN  214.556497         7.346   
4    6.0           0.0        78.89       2.759  212.412888         7.092   
5    4.0           0.0          NaN       2.756  126.160226         7.896   

     Year  Month   Day  day_of_week  hour  
0  2011.0    2.0  18.0          4.0   0.0  
1  2011.0    3.0  25.0          4.0   0.0  
3     NaN    NaN   NaN          NaN   NaN  
4  2010.0    5.0  28.0          4.0   0.0  
5  2010.0    5.0  28.0          4.0   0.0  


In [11]:
# Diviser le jeu de données en Train set & Test set
print("Dividing into train and test sets...")
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size=0.2, random_state=0)
print("...Done.")
print()

Dividing into train and test sets...
...Done.



In [12]:
# 1. Identification automatique/manuelle des types de variables
numeric_features = ['Temperature', 'Fuel_Price', 'CPI', 'Unemployment', 'Year', 'Month', 'Day', 'day_of_week']
categorical_features = ['Store', 'Holiday_Flag']

# 2. Création du pipeline pour les variables numériques
numeric_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='mean')), # Remplace les NaNs par la moyenne
    ('scaler', StandardScaler())                 # Obligatoire avant une régression régularisée !
])

# 3. Création du pipeline pour les variables catégorielles
categorical_transformer = Pipeline(steps=[
    ('imputer', SimpleImputer(strategy='most_frequent')), # Remplace les NaNs par la valeur la plus fréquente
    ('encoder', OneHotEncoder(drop='first', handle_unknown='ignore')) # Binarise et évite la colinéarité parfaite
])

# 4. Utilisation de ColumnTransformer pour regrouper les traitements
preprocessor = ColumnTransformer(transformers=[
    ('num', numeric_transformer, numeric_features),
    ('cat', categorical_transformer, categorical_features)
])

# 5. Preprocessings sur le train set
print("Performing preprocessings on train set...")
X_train_processed = preprocessor.fit_transform(X_train)
print('...Done.')

# 6. Preprocessings sur le test set
print("Performing preprocessings on test set...")
X_test_processed = preprocessor.transform(X_test) # STRICTEMENT transform() ! Ne jamais fit() sur le test set.
print('...Done.')

Performing preprocessings on train set...
...Done.
Performing preprocessings on test set...
...Done.


In [16]:
# Entraînement de la Régression Linéaire simple (sans régularisation)
print("Entraînement de la Régression Linéaire OLS...")
regressor = LinearRegression()

# On donne bien les données PROCESSED au modèle !
regressor.fit(X_train_processed, Y_train)
print("...Done.")

Entraînement de la Régression Linéaire OLS...
...Done.


In [17]:
# Prédictions sur le train et test set
Y_train_pred = regressor.predict(X_train_processed)
Y_test_pred = regressor.predict(X_test_processed)

# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))


R2 score on training set :  0.9731935046590678
R2 score on test set :  0.9344654257543307


In [18]:
# Validation croisée (3-fold) pour estimer le R2 généralisé avec Ridge
print("3-fold cross-validation...")
regressor_ridge = Ridge()

# Utilisation de X_train_processed (données standardisées) pour éviter le score négatif !
scores = cross_val_score(regressor_ridge, X_train_processed, Y_train, cv=3)

print('The cross-validated R2-score is : ', scores.mean())
print('The standard deviation is : ', scores.std())

3-fold cross-validation...
The cross-validated R2-score is :  0.8187495182769386
The standard deviation is :  0.028672961707481133


In [ ]:
# Print R^2 scores
print("R2 score on training set : ", r2_score(Y_train, Y_train_pred))
print("R2 score on test set : ", r2_score(Y_test, Y_test_pred))


R2 score on training set :  0.9777139215884865
R2 score on test set :  0.8889027368511802


In [ ]:
# Perform 3-fold cross-validation to evaluate the generalized R2 score obtained with a Ridge model
print("3-fold cross-validation...")
regressor = Ridge()
scores = cross_val_score(regressor, X_train, Y_train, cv=3)
print('The cross-validated R2-score is : ', scores.mean())
print('The standard deviation is : ', scores.std())


3-fold cross-validation...
The cross-validated R2-score is :  -0.24243818479736667
The standard deviation is :  0.20643506428528294


In [19]:
# Perform grid search pour trouver le meilleur coefficient de régularisation
print("Grid search...")
regressor_ridge = Ridge()

# Grille de valeurs à tester (issue du cours Jedha)
params = {
    'alpha': [0.01, 0.05, 0.1, 0.5, 1, 5, 10, 50, 100]
}

# Paramétrage de la recherche avec validation croisée intégrée (cv=5)
best_ridge = GridSearchCV(regressor_ridge, param_grid=params, cv=5)

# Entraînement systématique sur toutes les valeurs d'alpha
best_ridge.fit(X_train_processed, Y_train)
print("...Done.")

print("Best hyperparameters : ", best_ridge.best_params_)
print("Best R2 score (Cross-validated) : ", best_ridge.best_score_)


Grid search...
...Done.
Best hyperparameters :  {'alpha': 0.01}
Best R2 score (Cross-validated) :  0.9446168427355286


In [20]:
# Évaluation finale du meilleur modèle trouvé par le GridSearch
print("--- SCORES FINAUX (MODELE OPTIMISE) ---")
print("RIDGE / R2 score on training set : ", best_ridge.score(X_train_processed, Y_train))
print("RIDGE / R2 score on test set : ", best_ridge.score(X_test_processed, Y_test))

--- SCORES FINAUX (MODELE OPTIMISE) ---
RIDGE / R2 score on training set :  0.9731756355275908
RIDGE / R2 score on test set :  0.9358630190131586
